# 02 Baseline Models

Compare the initial models on periods 29-35 after training through period 28. Report: Section 3.1. Neural models use an inner epoch-selection split followed by a fresh refit.


## 1. Setup
Set the data path, choose a runtime and install the dependencies used below. The `quick` profile checks the workflow with a small budget; use `report` for the report experiments. Place the four original CSV files in one folder.


In [ ]:
from pathlib import Path
# Use the same data directory and run name in all notebooks.
PROFILE = globals().get('PROFILE', 'quick')  # quick: smoke test; report: full training
DATA_SOURCE = globals().get('DATA_SOURCE', 'drive')  # drive, upload or local
DATA_DIR = Path(globals().get('DATA_DIR', '/content/drive/MyDrive/bt-4012-competition-2026'))
OUTPUT_ROOT = Path(globals().get('OUTPUT_ROOT', '/content/drive/MyDrive/BT4012_Final_Report_English'))
RUN_NAME = globals().get('RUN_NAME', 'final_report_english_v1')
INSTALL_DEPENDENCIES = globals().get('INSTALL_DEPENDENCIES', True)
EXPECTED_COMPETITION = globals().get('EXPECTED_COMPETITION', True)
assert PROFILE in ['quick', 'report']
OUT = OUTPUT_ROOT / RUN_NAME / PROFILE


In [ ]:
import sys, subprocess
# Keep the torch/CUDA version supplied by Colab.
PACKAGES = ['numpy>=1.26', 'pandas>=2.0', 'scipy>=1.11', 'scikit-learn>=1.6', 'joblib', 'tabm==0.0.3', 'rtdl-num-embeddings==0.0.12', 'rtdl-revisiting-models==0.0.2', 'xgboost>=2.0', 'lightgbm>=4.0', 'catboost==1.2.10']
if INSTALL_DEPENDENCIES:
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', *PACKAGES], check=True)


In [ ]:
CSV_FILES = ['train.csv', 'test.csv', 'txs_edgelist.csv', 'sample_submission.csv']

if DATA_SOURCE == 'drive':
    from google.colab import drive
    drive.mount('/content/drive')
elif DATA_SOURCE == 'upload':
    from google.colab import files
    DATA_DIR = Path('/content/bt4012_data')
    DATA_DIR.mkdir(exist_ok=True)
    missing = [name for name in CSV_FILES if not (DATA_DIR / name).exists()]
    if missing:
        print('Upload the missing files:', missing)
        for name, content in files.upload().items():
            if name not in missing:
                raise ValueError('Unexpected file: ' + name)
            (DATA_DIR / name).write_bytes(content)
    if str(OUTPUT_ROOT).startswith('/content/drive/'):
        OUTPUT_ROOT = Path('/content/BT4012_Final_Report_English')
        OUT = OUTPUT_ROOT / RUN_NAME / PROFILE
elif DATA_SOURCE != 'local':
    raise ValueError(DATA_SOURCE)
for name in CSV_FILES:
    if not (DATA_DIR / name).is_file():
        raise FileNotFoundError(DATA_DIR / name)
OUT.mkdir(parents=True, exist_ok=True)
print('Mode:', PROFILE, '| Output:', OUT)


## 2. Required functions
Only the data structures, models and helpers used by this notebook are included below.


### TabR attribution
The TabR implementation is adapted from https://github.com/yandex-research/tabular-dl-tabr.

```text
Copyright (c) 2023 Authors of "TabR: Unlocking the Power of Retrieval-Augmented Tabular Deep Learning"

Permission is hereby granted, free of charge, to any person obtaining
a copy of this software and associated documentation files (the
"Software"), to deal in the Software without restriction, including
without limitation the rights to use, copy, modify, merge, publish,
distribute, sublicense, and/or sell copies of the Software, and to
permit persons to whom the Software is furnished to do so, subject to
the following conditions:
The above copyright notice and this permission notice shall be
included in all copies or substantial portions of the Software.
THE SOFTWARE IS PROVIDED "AS IS", WITHOUT WARRANTY OF ANY KIND,
EXPRESS OR IMPLIED, INCLUDING BUT NOT LIMITED TO THE WARRANTIES OF
MERCHANTABILITY, FITNESS FOR A PARTICULAR PURPOSE AND
NONINFRINGEMENT. IN NO EVENT SHALL THE AUTHORS OR COPYRIGHT HOLDERS BE
LIABLE FOR ANY CLAIM, DAMAGES OR OTHER LIABILITY, WHETHER IN AN ACTION
OF CONTRACT, TORT OR OTHERWISE, ARISING FROM, OUT OF OR IN CONNECTION
WITH THE SOFTWARE OR THE USE OR OTHER DEALINGS IN THE SOFTWARE.
```


In [ ]:
from sklearn.ensemble import ExtraTreesClassifier
from torch.nn import functional as F
from sklearn.model_selection import GroupShuffleSplit
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.linear_model import LinearRegression
from sklearn.linear_model import LogisticRegression
from pathlib import Path
from sklearn.preprocessing import QuantileTransformer
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import Ridge
from sklearn.linear_model import RidgeClassifier
from sklearn.preprocessing import RobustScaler
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import average_precision_score
from scipy.sparse.csgraph import connected_components
from scipy.sparse import coo_matrix
from dataclasses import dataclass
from collections import deque
import gc
import hashlib
import joblib
import json
from torch import nn
import numpy as np
import pandas as pd
import random
from sklearn.metrics import roc_auc_score
import time
import torch
from sklearn.model_selection import train_test_split
import os
torch.set_num_threads(min(4, os.cpu_count() or 1))

VERSION = 'bt4012-colab-report-20261021-v1'

ENGINE_HASH = '3661498eb4e7e545253aeae84b480c6d4c349dc7e125920bf97989a5e96c0840'

FEATURES = [f'feat_{i}' for i in range(1, 166)]

TABM = dict(k=32, n_blocks=2, d_block=512, dropout=0.1, n_bins=48, d_embedding=16, learning_rate=0.002, weight_decay=0.0003)

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

def serial(x):
    if isinstance(x, (np.integer, np.floating)):
        return x.item()
    if isinstance(x, np.ndarray):
        return x.tolist()
    if isinstance(x, Path):
        return str(x)
    raise TypeError(type(x).__name__)

def save_json(path, value):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    temp = path.with_suffix(path.suffix + '.tmp')
    temp.write_text(json.dumps(value, ensure_ascii=False, indent=2, default=serial, allow_nan=False))
    temp.replace(path)

def fingerprint(value):
    return hashlib.sha256(json.dumps(value, sort_keys=True, default=serial).encode()).hexdigest()

def array_hash(x):
    return hashlib.sha256(np.ascontiguousarray(x).tobytes()).hexdigest()

def file_hash(path):
    h = hashlib.sha256()
    with open(path, 'rb') as f:
        for block in iter(lambda: f.read(8 * 1024 ** 2), b''):
            h.update(block)
    return h.hexdigest()

def seed_all(seed):
    """Set random seeds before constructing and training a model."""
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.benchmark = False
    torch.backends.cudnn.deterministic = True

def metrics(y, p):
    y = np.asarray(y)
    p = np.asarray(p)
    if len(y) != len(p) or not len(y) or (not np.isfinite(p).all()):
        raise ValueError('Empty, nonfinite or unaligned predictions')
    if not np.isin(y, [0, 1]).all():
        raise ValueError('Unknown labels cannot be scored')
    return dict(n=len(y), positives=int(y.sum()), auc=float(roc_auc_score(y, p)) if len(np.unique(y)) == 2 else None, ap=float(average_precision_score(y, p)) if len(np.unique(y)) == 2 else None)

def profiles(mode):
    """Use a small budget for a smoke test or the report budget for full experiments."""
    if mode not in ['quick', 'report']:
        raise ValueError(mode)
    return dict(mode=mode, seeds=[42] if mode == 'quick' else [17, 42, 73], cutoffs=[21] if mode == 'quick' else [7, 10, 14, 17, 21], max_epochs=2 if mode == 'quick' else 100, patience=1 if mode == 'quick' else 10, trees=30 if mode == 'quick' else None, batch_size=256)

class Dataset:
    """Read the four CSV files and build the transaction graph."""

    def __init__(self, data_dir, expected_competition=True):
        self.data_dir = Path(data_dir)
        self.train = pd.read_csv(self.data_dir / 'train.csv')
        self.test = pd.read_csv(self.data_dir / 'test.csv')
        self.sample = pd.read_csv(self.data_dir / 'sample_submission.csv')
        raw_edges = pd.read_csv(self.data_dir / 'txs_edgelist.csv')
        self.duplicate_edges = int(raw_edges.duplicated().sum())
        self.edges = raw_edges.drop_duplicates(['txId1', 'txId2']).reset_index(drop=True)
        if list(self.train.columns) != ['txId', 'time_step', *FEATURES, 'label']:
            raise ValueError('Unexpected train columns/order')
        if list(self.test.columns) != ['index', 'txId', 'time_step', *FEATURES]:
            raise ValueError('Unexpected test columns/order')
        if list(self.sample.columns) != ['index', 'target']:
            raise ValueError('Submission schema')
        if self.train.txId.duplicated().any() or self.test.txId.duplicated().any():
            raise ValueError('Duplicate IDs')
        if set(self.train.txId) & set(self.test.txId):
            raise ValueError('Overlapping IDs')
        if not np.array_equal(self.sample['index'], self.test['index']):
            raise ValueError('Submission order')
        if not self.test['index'].is_unique:
            raise ValueError('Duplicate submission IDs')
        if not self.train.label.dropna().isin([0, 1]).all():
            raise ValueError('Labels must be 0/1/blank')
        for df in [self.train, self.test]:
            if not np.isfinite(df[FEATURES].to_numpy()).all():
                raise ValueError('Nonfinite raw features')
        if expected_competition:
            assert (len(self.train), len(self.test), len(self.edges)) == (141772, 15329, 234355)
            assert self.train.time_step.between(1, 35).all() and self.test.time_step.between(36, 49).all()
        self.ids = np.unique(np.r_[self.train.txId, self.test.txId, self.edges.to_numpy().ravel()])
        self.src = np.searchsorted(self.ids, self.edges.txId1)
        self.dst = np.searchsorted(self.ids, self.edges.txId2)
        n = len(self.ids)
        self.outgoing = coo_matrix((np.ones(len(self.src)), (self.src, self.dst)), shape=(n, n)).tocsr()
        self.incoming = self.outgoing.T.tocsr()
        self.indeg = np.diff(self.incoming.indptr)
        self.outdeg = np.diff(self.outgoing.indptr)
        self.nc, self.components = connected_components(self.outgoing, directed=False)
        all_rows = pd.concat([self.train, self.test], ignore_index=True).set_index('txId').reindex(self.ids)
        self.node_raw = all_rows[FEATURES].to_numpy(np.float32)
        self.has_features = np.isfinite(self.node_raw).all(1)
        self.node_y = all_rows.label.fillna(-1).to_numpy(np.int64)
        observed = all_rows.time_step.to_numpy(float)
        self.steps = np.full(n, -1, np.int64)
        for c in range(self.nc):
            member = self.components == c
            times = np.unique(observed[member & np.isfinite(observed)])
            if len(times) != 1:
                raise ValueError('Component does not identify exactly one snapshot')
            self.steps[member] = int(times[0])
        if np.any(self.steps[self.src] != self.steps[self.dst]):
            raise ValueError('Cross-snapshot edge')
        q = deque(np.flatnonzero(self.indeg == 0).tolist())
        deg = self.indeg.copy()
        self.order = []
        while q:
            v = q.popleft()
            self.order.append(v)
            for w in self.outgoing.indices[self.outgoing.indptr[v]:self.outgoing.indptr[v + 1]]:
                deg[w] -= 1
                if deg[w] == 0:
                    q.append(int(w))
        if len(self.order) != n:
            raise ValueError('Graph is not a DAG')
        # Unknown labels are graph context, not negative examples.
        self.frame = self.train.loc[self.train.label.notna()].copy().reset_index(drop=True)
        self.frame.label = self.frame.label.astype(np.int64)
        self.pos = np.searchsorted(self.ids, self.frame.txId)
        self.testpos = np.searchsorted(self.ids, self.test.txId)
        self.node_x = np.column_stack([self.node_raw, self.steps, self.indeg, self.outdeg]).astype(np.float32)
        self.raw = self.node_x[self.pos]
        self.y = self.frame.label.to_numpy(np.int64)
        self._extras = {}
        self.data_hashes = {name: file_hash(self.data_dir / name) for name in ['train.csv', 'test.csv', 'txs_edgelist.csv', 'sample_submission.csv']}
        self.identity = fingerprint(self.data_hashes)

    def split(self, cut, horizon=14):
        t = self.frame.time_step.to_numpy()
        a = np.flatnonzero(t <= cut)
        b = np.flatnonzero((t > cut) & (t <= cut + horizon))
        if not len(a) or not len(b):
            raise ValueError('Empty temporal split')
        return (a, b)


In [ ]:
def prepare_num(A, B, scaler='quantile'):
    """Fit the imputer, constant-column mask and scaler using training rows only."""
    # Fit every transformation using the current training fold.
    imputer = SimpleImputer(strategy='median', keep_empty_features=True)
    X = imputer.fit_transform(A).astype(np.float32)
    V = imputer.transform(B).astype(np.float32)
    keep = np.ptp(X, axis=0) > 0
    if not keep.any():
        raise ValueError('All numerical inputs constant')
    X = X[:, keep]
    V = V[:, keep]
    if scaler == 'quantile':
        transform = QuantileTransformer(n_quantiles=max(10, min(len(X) // 30, 1000)), output_distribution='normal', subsample=10 ** 9, random_state=0)
        noise = np.random.default_rng(0).normal(0, 1e-05, X.shape).astype(np.float32)
        transform.fit(X + noise)
    elif scaler == 'robust':
        transform = RobustScaler().fit(X)
    elif scaler == 'standard':
        transform = StandardScaler().fit(X)
    elif scaler == 'none':
        transform = None
    else:
        raise ValueError(scaler)
    bundle = (imputer, keep, transform)
    return (apply_num(A, bundle), apply_num(B, bundle), bundle)

def apply_num(X, bundle):
    """Apply a saved numeric transformation without refitting it."""
    imputer, keep, transform = bundle
    X = imputer.transform(X).astype(np.float32)[:, keep]
    return (transform.transform(X) if transform is not None else X).astype(np.float32)

def category_codes(x, levels):
    """Use zero for categories not observed in the training fold."""
    ix = np.searchsorted(levels, x)
    good = ix < len(levels)
    good[good] &= np.asarray(levels)[ix[good]] == x[good]
    return np.where(good, ix + 1, 0).astype(np.int64)

def make_inputs(data, fit, query, variant='raw', scaler='quantile', base='tabm', test=False, genes=None, feature_plan=None):
    """Fit numeric preprocessing on training rows and apply it to the requested rows."""
    if variant not in ['raw', 'cat3']:
        raise ValueError(variant)
    columns = 168 if base == 'tabm' else 166
    A = data.raw[fit, :columns].copy()
    B = (data.node_x[data.testpos] if test else data.raw[query])[:, :columns].copy()
    C = np.empty((len(A), 0), dtype=np.int64)
    D = np.empty((len(B), 0), dtype=np.int64)
    cardinalities = []
    levels = None
    if variant == 'cat3':
        levels = np.unique(A[:, 2])
        C = category_codes(A[:, 2], levels)[:, None]
        D = category_codes(B[:, 2], levels)[:, None]
        A = np.delete(A, 2, axis=1)
        B = np.delete(B, 2, axis=1)
        cardinalities = [len(levels) + 1]
    X, V, prep = prepare_num(A, B, scaler)
    state = dict(variant=variant, base=base, fit_ids_hash=array_hash(data.frame.txId.to_numpy()[fit]), scaler=scaler, numerical_preprocessor=prep, cardinalities=cardinalities)
    if levels is not None:
        state['levels'] = levels
    return (X, V, C, D, state)

def tree_model(name, y, seed, profile):
    """Construct the selected tree or linear baseline with fixed parameters."""
    w = float((y == 0).sum() / max((y == 1).sum(), 1))
    small = profile['trees']
    if name == 'logistic':
        return LogisticRegression(C=1, class_weight='balanced', max_iter=2000, random_state=seed)
    if name == 'linear':
        return LinearRegression()
    if name == 'ridge':
        return Ridge(alpha=1)
    if name == 'ridge_classifier':
        return RidgeClassifier(alpha=1, class_weight='balanced')
    if name in ['random_forest', 'extra_trees']:
        cls = RandomForestClassifier if name == 'random_forest' else ExtraTreesClassifier
        return cls(n_estimators=small or 500, min_samples_leaf=2, max_features='sqrt', class_weight='balanced', n_jobs=2, random_state=seed)
    if name == 'hist_gradient_boosting':
        return HistGradientBoostingClassifier(max_iter=small or 500, learning_rate=0.05, max_leaf_nodes=31, min_samples_leaf=30, l2_regularization=1, class_weight='balanced', early_stopping=False, random_state=seed)
    if name == 'xgboost':
        from xgboost import XGBClassifier
        return XGBClassifier(n_estimators=small or 1200, learning_rate=0.025, max_depth=6, min_child_weight=5, subsample=0.85, colsample_bytree=0.85, reg_alpha=0.1, reg_lambda=1, scale_pos_weight=w, tree_method='hist', n_jobs=2, random_state=seed)
    if name == 'lightgbm':
        from lightgbm import LGBMClassifier
        return LGBMClassifier(n_estimators=small or 600, learning_rate=0.025, num_leaves=31, min_child_samples=40, subsample=0.85, colsample_bytree=0.85, reg_alpha=0.1, reg_lambda=1, scale_pos_weight=w, n_jobs=2, random_state=seed, verbosity=-1)
    if name == 'catboost':
        from catboost import CatBoostClassifier
        return CatBoostClassifier(iterations=small or 1200, learning_rate=0.025, depth=6, l2_leaf_reg=3, class_weights=[1, w], thread_count=2, random_seed=seed, verbose=False, allow_writing_files=False)
    raise ValueError(name)

class FeatureLSTM(nn.Module):

    def __init__(self, d):
        super().__init__()
        self.lstm = nn.LSTM(1, 32, batch_first=True)
        self.head = nn.Sequential(nn.Linear(33, 32), nn.ReLU(), nn.Dropout(0.2), nn.Linear(32, 1))

    def forward(self, x):
        _, (h, _) = self.lstm(x[:, :-1, None])
        return self.head(torch.cat([h[-1], x[:, -1:]], 1))

def build_net(name, X, cardinalities, config):
    """Construct a neural model from random initial weights."""
    if name == 'tabm':
        import tabm, rtdl_num_embeddings as emb
        cfg = dict(config)
        cfg.pop('learning_rate')
        cfg.pop('weight_decay')
        bins = emb.compute_bins(torch.from_numpy(X), n_bins=cfg.pop('n_bins'))
        module = emb.PiecewiseLinearEmbeddings(bins, cfg.pop('d_embedding'), activation=False, version='B')
        return tabm.TabM.make(n_num_features=X.shape[1], cat_cardinalities=cardinalities, d_out=2, num_embeddings=module, **cfg).to(DEVICE)
    if name == 'tabr':
        return TabR(X.shape[1]).to(DEVICE)
    if name == 'lstm':
        return FeatureLSTM(X.shape[1]).to(DEVICE)
    import rtdl_revisiting_models as rtdl
    if name == 'mlp':
        return rtdl.MLP(d_in=X.shape[1], d_out=1, n_blocks=3, d_block=256, dropout=0.2).to(DEVICE)
    if name == 'resnet':
        return rtdl.ResNet(d_in=X.shape[1], d_out=1, n_blocks=3, d_block=128, d_hidden_multiplier=2, dropout1=0.2, dropout2=0).to(DEVICE)
    if name == 'ft_transformer':
        settings = dict(n_blocks=2, d_block=128, attention_n_heads=8, attention_dropout=0.15, ffn_d_hidden_multiplier=4 / 3, ffn_dropout=0.05, residual_dropout=0.0) if cardinalities else rtdl.FTTransformer.get_default_kwargs(n_blocks=2)
        return rtdl.FTTransformer(n_cont_features=X.shape[1], cat_cardinalities=cardinalities, d_out=2 if cardinalities else 1, **settings).to(DEVICE)
    raise ValueError(name)

def head_loss(logits, y):
    """Average cross-entropy across all TabM members and training examples."""
    return F.cross_entropy(logits.flatten(0, 1), y.repeat_interleave(logits.shape[1]))

def forward_net(net, name, x, c, pool, y, idx=None):
    if name in ['tabm', 'ft_transformer']:
        return net(x, c if c.shape[1] else None)
    if name == 'tabr':
        return net(x, pool, y, self_indices=idx)
    return net(x)

@torch.inference_mode()
def net_predict(net, name, V, D, pool, y, batch=256):
    """Run inference and average the 32 member probabilities for TabM."""
    net.eval()
    parts = []
    for start in range(0, len(V), batch):
        x = torch.as_tensor(V[start:start + batch], device=DEVICE)
        c = torch.as_tensor(D[start:start + batch], device=DEVICE)
        logits = forward_net(net, name, x, c, pool, y)
        p = logits.softmax(-1)[..., 1].mean(1) if name == 'tabm' else logits.softmax(-1)[:, 1] if logits.ndim == 2 and logits.shape[1] == 2 else logits.reshape(-1).sigmoid()
        parts.append(p.cpu().numpy())
    return np.concatenate(parts)

def fit_net(name, seed, X, C, y, V, D, profile, config=None, val_y=None, epochs=None):
    """Select an epoch count on inner validation, or refit for a fixed number of epochs."""
    seed_all(seed)
    cfg = dict(TABM if config is None else config)
    net = build_net(name, X, [] if C.shape[1] == 0 else [int(max(C.max(), D.max(initial=0))) + 1], cfg)
    tx = torch.as_tensor(X, device=DEVICE)
    tc = torch.as_tensor(C, device=DEVICE)
    ty = torch.as_tensor(y, device=DEVICE)
    lr = cfg['learning_rate'] if name == 'tabm' else 0.001 if name != 'ft_transformer' else 0.0001
    opt = torch.optim.AdamW(net.parameters(), lr=lr, weight_decay=cfg['weight_decay'] if name == 'tabm' else 0.0001)
    curve = []
    best = -np.inf
    selected = 0
    stale = 0
    for epoch in range(1, (epochs or profile['max_epochs']) + 1):
        net.train()
        loss_sum = 0.0
        for idx in torch.randperm(len(tx), device=DEVICE).split(profile['batch_size']):
            opt.zero_grad(set_to_none=True)
            logits = forward_net(net, name, tx[idx], tc[idx], tx, ty, idx)
            if name == 'tabm':
                loss = head_loss(logits, ty[idx])
            elif logits.ndim == 2 and logits.shape[1] == 2:
                loss = F.cross_entropy(logits, ty[idx])
            else:
                weight = (ty == 0).sum() / torch.clamp((ty == 1).sum(), min=1) if name in ['mlp', 'resnet', 'ft_transformer', 'lstm'] else None
                loss = F.binary_cross_entropy_with_logits(logits.reshape(-1), ty[idx].float(), pos_weight=weight)
            if not torch.isfinite(loss):
                raise ValueError('Nonfinite training loss')
            loss.backward()
            nn.utils.clip_grad_norm_(net.parameters(), 1.0)
            opt.step()
            loss_sum += float(loss.detach()) * len(idx)
        row = dict(epoch=epoch, loss=loss_sum / len(tx))
        if epochs is None:
            if val_y is None:
                raise ValueError('Selection needs inner labels, never outer labels')
            row.update(metrics(val_y, net_predict(net, name, V, D, tx, ty)))
            if row['auc'] is None:
                raise ValueError('Inner selection window has one class')
            if row['auc'] > best:
                best = row['auc']
                selected = epoch
                stale = 0
            else:
                stale += 1
        else:
            selected = epoch
        curve.append(row)
        if epochs is None and stale >= profile['patience']:
            break
    p = net_predict(net, name, V, D, tx, ty)
    checkpoint = dict(state_dict={k: v.detach().cpu().clone() for k, v in net.state_dict().items()}, config=cfg, name=name, input_dim=X.shape[1], cardinalities=[] if C.shape[1] == 0 else [int(C.max()) + 1], epochs=selected, seed=seed, initialization='random', pretrained=False)
    del net, opt, tx, tc, ty
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
    return (p, selected, curve, checkpoint)


In [ ]:
def predict_tree(model, X):
    if hasattr(model, 'predict_proba'):
        return model.predict_proba(X)[:, 1]
    if hasattr(model, 'decision_function'):
        return model.decision_function(X)
    return model.predict(X)

def split_indices(data, cut, horizon=14, kind='time', split_seed=20261016, inner_horizon=3):
    """Keep the epoch-selection split inside the outer training history."""
    if kind == 'time':
        a, b = data.split(cut, horizon)
        innercut = cut - min(inner_horizon, cut - 4)
        ia, ib = data.split(innercut, cut - innercut)
    elif kind == 'random':
        a, b = train_test_split(np.arange(len(data.frame)), test_size=0.3, stratify=data.y, random_state=split_seed)
        ia, ib = train_test_split(a, test_size=0.3, stratify=data.y[a], random_state=split_seed + 1)
    elif kind == 'component':
        groups = data.components[data.pos]
        ix = np.arange(len(data.frame))
        a, b = next(GroupShuffleSplit(n_splits=1, test_size=0.3, random_state=split_seed).split(ix, groups=groups))
        x, z = next(GroupShuffleSplit(n_splits=1, test_size=0.3, random_state=split_seed + 1).split(a, groups=groups[a]))
        ia, ib = (a[x], a[z])
    else:
        raise ValueError(kind)
    for z in [a, b, ia, ib]:
        if len(np.unique(data.y[z])) < 2:
            raise ValueError('Split needs both labels')
    assert not np.intersect1d(a, b).size and set(ia) | set(ib) <= set(a)
    return tuple((np.sort(z) for z in [a, b, ia, ib]))

def run_trial(data, out, model='tabm', variant='raw', cut=21, horizon=14, seed=42, profile=None, config=None, inner_horizon=3, base='tabm', kind='time', split_seed=20261016, scaler=None, stage='experiments', genes=None, epoch_policy='own', control_variant='raw', feature_plan=None):
    """Fit one historical trial without using outer validation labels for epoch selection."""
    profile = profile or profiles('quick')
    config = dict(TABM if config is None else config)
    neural = model in ['tabm', 'tabr', 'mlp', 'resnet', 'ft_transformer', 'lstm']
    scaler = scaler or ('quantile' if neural else 'standard' if model in ['logistic', 'linear', 'ridge', 'ridge_classifier'] else 'none')
    spec = dict(version=VERSION, data=data.identity, stage=stage, model=model, variant=variant, cut=cut, horizon=horizon, seed=seed, profile=profile, config=config, inner_horizon=inner_horizon, base=base, kind=kind, split_seed=split_seed, scaler=scaler, genes=genes, epoch_policy=epoch_policy, control_variant=control_variant, feature_plan=feature_plan, engine_hash=globals().get('ENGINE_HASH', VERSION))
    key = fingerprint(spec)
    job = Path(out) / stage / f'{model}_{variant}_{key[:16]}'
    if (job / 'complete.json').exists():
        record = json.loads((job / 'complete.json').read_text())
        assert record['spec'] == json.loads(json.dumps(spec))
        p = pd.read_csv(job / 'predictions.csv', float_precision='round_trip')
        assert array_hash(p.txId.to_numpy()) == record['query_ids_hash']
        assert metrics(p.label, p.prediction) == record['metrics']
        print('REUSE', model, variant, cut, seed, flush=True)
        return record
    a, b, ia, ib = split_indices(data, cut, horizon, kind, split_seed, inner_horizon)
    job.mkdir(parents=True, exist_ok=True)
    save_json(job / 'started.json', spec)
    start = time.time()
    X, V, C, D, bundle = make_inputs(data, a, b, variant, scaler, base, genes=genes, feature_plan=feature_plan)
    if neural:
        selection_variant = control_variant if epoch_policy == 'control' else variant
        A, B, E, G, inner = make_inputs(data, ia, ib, selection_variant, scaler, base, genes=genes, feature_plan=feature_plan)
        _, ep, selection, _ = fit_net(model, seed, A, E, data.y[ia], B, G, profile, config, val_y=data.y[ib])
        p, _, curve, weights = fit_net(model, seed, X, C, data.y[a], V, D, profile, config, epochs=ep)
        torch.save(weights, job / 'model.pt')
        joblib.dump(dict(outer=bundle, inner=inner), job / 'preprocessing.joblib')
        save_json(job / 'selection.json', selection)
        save_json(job / 'refit_curve.json', curve)
    else:
        ep = None
        estimator = tree_model(model, data.y[a], seed, profile)
        if C.shape[1]:
            levels = bundle['cardinalities'][0]
            X = np.column_stack([X, np.eye(levels)[C[:, 0]]])
            V = np.column_stack([V, np.eye(levels)[D[:, 0]]])
        estimator.fit(X, data.y[a])
        p = predict_tree(estimator, V)
        joblib.dump(dict(estimator=estimator, preprocessor=bundle), job / 'model.joblib')
    pred = data.frame.iloc[b][['txId', 'time_step', 'label']].copy()
    pred['prediction'] = np.asarray(p, dtype=np.float64)
    pred.to_csv(job / 'predictions.csv', index=False)
    record = dict(spec=spec, metrics=metrics(data.y[b], p), selected_epochs=ep, train_rows=len(a), train_ids_hash=array_hash(data.frame.txId.to_numpy()[a]), query_ids_hash=array_hash(pred.txId.to_numpy()), prediction_hash=file_hash(job / 'predictions.csv'), seconds=time.time() - start, path=str(job), outer_labels_used_for_epoch_selection=False, complete=True)
    save_json(job / 'complete.json', record)
    print('DONE', model, variant, cut, seed, record['metrics'], flush=True)
    return record

def prediction_records(out, stage=None):
    records = []
    for f in Path(out).rglob('complete.json'):
        r = json.loads(f.read_text())
        if all((k in r for k in ['spec', 'path', 'metrics', 'query_ids_hash'])) and 'stage' in r['spec'] and (r['spec'].get('engine_hash') == globals().get('ENGINE_HASH', VERSION)) and (stage is None or r['spec']['stage'] == stage):
            records.append(r)
    return records

def aggregate_runs(records):
    """Average seed predictions before calculating each window metric."""
    groups = {}
    for r in records:
        s = r['spec']
        g = {k: v for k, v in s.items() if k != 'seed'}
        key = fingerprint(g)
        groups.setdefault(key, (g, []))[1].append(r)
    rows = []
    for spec, rs in groups.values():
        rs = sorted(rs, key=lambda r: r['spec']['seed'])
        seeds = [r['spec']['seed'] for r in rs]
        if len(seeds) != len(set(seeds)):
            raise ValueError('Duplicate seeded trials')
        expected = spec['profile']['seeds']
        complete = sorted(seeds) == sorted(expected)
        reference = None
        parts = []
        for r in rs:
            frame = pd.read_csv(Path(r['path']) / 'predictions.csv', float_precision='round_trip')
            if reference is None:
                reference = frame
            else:
                np.testing.assert_array_equal(reference[['txId', 'time_step', 'label']], frame[['txId', 'time_step', 'label']])
            parts.append(frame.prediction.to_numpy())
        bag = np.column_stack(parts).mean(1)
        row = dict(stage=spec['stage'], model=spec['model'], variant=spec['variant'], cut=spec['cut'], horizon=spec['horizon'], kind=spec['kind'], config_id=fingerprint(spec['config'])[:10], inner_horizon=spec['inner_horizon'], scaler=spec['scaler'], base=spec['base'], complete=complete, seeds=seeds, **metrics(reference.label, bag))
        rows.append(row)
    return pd.DataFrame(rows)

def nearest_indices(query_keys, pool_keys, size, self_indices=None):
    """Label-independent exact search. Exclude the query's own pool row."""
    with torch.no_grad():
        distances = query_keys.square().sum(1, keepdim=True) + pool_keys.square().sum(1)[None] - 2 * query_keys @ pool_keys.T
        if self_indices is not None:
            distances[torch.arange(len(query_keys), device=query_keys.device), self_indices] = torch.inf
        eligible = len(pool_keys) - int(self_indices is not None)
        if eligible < 1:
            raise ValueError('Retrieval needs a non-self candidate')
        return distances.topk(min(size, eligible), largest=False, sorted=True).indices

class TabR(nn.Module):

    def __init__(self, n_features, d_main=128, predictor_n_blocks=2, context_size=96, dropout=0.1, context_dropout=0.2):
        super().__init__()
        d_block = 2 * d_main
        self.linear = nn.Linear(n_features, d_main)
        self.K = nn.Linear(d_main, d_main)
        self.label_encoder = nn.Embedding(2, d_main)
        nn.init.uniform_(self.label_encoder.weight, -1, 1)
        self.T = nn.Sequential(nn.Linear(d_main, d_block), nn.ReLU(), nn.Dropout(dropout), nn.Linear(d_block, d_main, bias=False))
        self.dropout = nn.Dropout(context_dropout)
        self.blocks1 = nn.ModuleList([nn.Sequential(nn.LayerNorm(d_main), nn.Linear(d_main, d_block), nn.ReLU(), nn.Dropout(dropout), nn.Linear(d_block, d_main), nn.Dropout(0.0)) for _ in range(predictor_n_blocks)])
        self.head = nn.Sequential(nn.LayerNorm(d_main), nn.ReLU(), nn.Linear(d_main, 1))
        self.context_size = context_size

    def encode(self, x):
        value = self.linear(x)
        return (value, self.K(value))

    def forward(self, x, pool_x, pool_y, self_indices=None, pool_keys=None):
        if self.training and pool_keys is not None:
            raise ValueError('Stale training keys are forbidden')
        if self.training and self_indices is None:
            raise ValueError('Training queries must have explicit self indices')
        if pool_keys is None:
            with torch.no_grad():
                pool_keys = self.encode(pool_x)[1]
        value, keys = self.encode(x)
        indices = nearest_indices(keys.detach(), pool_keys, self.context_size, self_indices)
        if torch.is_grad_enabled():
            context_keys = self.encode(pool_x[indices].flatten(0, 1))[1].reshape(len(x), indices.shape[1], -1)
        else:
            context_keys = pool_keys[indices]
        similarities = -keys.square().sum(-1, keepdim=True) + 2 * (keys[:, None] @ context_keys.transpose(-1, -2)).squeeze(1) - context_keys.square().sum(-1)
        probabilities = self.dropout(similarities.softmax(-1))
        values = self.label_encoder(pool_y[indices]) + self.T(keys[:, None] - context_keys)
        value = value + (probabilities[:, None] @ values).squeeze(1)
        for block in self.blocks1:
            value = value + block(value)
        return self.head(value).squeeze(-1)


In [ ]:
def neighbor_matrix(src, dst, n: int, upstream: bool, device):
    rows, cols = (dst, src) if upstream else (src, dst)
    degree = np.bincount(rows, minlength=n)
    values = (1.0 / degree[rows]).astype(np.float32)
    indices = torch.from_numpy(np.stack([rows, cols])).long().to(device)
    return torch.sparse_coo_tensor(indices, torch.from_numpy(values).to(device), (n, n), device=device).coalesce()

@dataclass
class SnapshotBatch:
    x: torch.Tensor
    y: torch.Tensor
    labeled: torch.Tensor
    incoming: torch.Tensor
    outgoing: torch.Tensor
    tx_ids: np.ndarray
    steps: np.ndarray
    edge_count: int

def make_batch(mask, x, y, steps, ids, src, dst, device):
    chosen = np.flatnonzero(mask)
    mapping = np.full(len(ids), -1, dtype=np.int64)
    mapping[chosen] = np.arange(len(chosen))
    crossing = mask[src] != mask[dst]
    if crossing.any():
        raise ValueError('An edge crosses the time partition')
    keep = mask[src] & mask[dst]
    local_src, local_dst = (mapping[src[keep]], mapping[dst[keep]])
    return SnapshotBatch(x=torch.from_numpy(x[chosen]).to(device), y=torch.from_numpy(y[chosen].astype(np.float32)).to(device), labeled=torch.from_numpy(y[chosen] >= 0).to(device), incoming=neighbor_matrix(local_src, local_dst, len(chosen), True, device), outgoing=neighbor_matrix(local_src, local_dst, len(chosen), False, device), tx_ids=ids[chosen], steps=steps[chosen], edge_count=int(keep.sum()))

class DirectedMeanLayer(nn.Module):

    def __init__(self, in_features: int, out_features: int):
        super().__init__()
        self.self_linear = nn.Linear(in_features, out_features)
        self.in_linear = nn.Linear(in_features, out_features, bias=False)
        self.out_linear = nn.Linear(in_features, out_features, bias=False)
        self.norm = nn.LayerNorm(out_features)

    def forward(self, x, incoming, outgoing):
        h = self.self_linear(x)
        h = h + self.in_linear(torch.sparse.mm(incoming, x))
        h = h + self.out_linear(torch.sparse.mm(outgoing, x))
        return F.relu(self.norm(h))

class DirectedGraphSAGE(nn.Module):

    def __init__(self, input_dim: int, hidden: int, dropout: float, layers: int=2):
        super().__init__()
        if layers < 2:
            raise ValueError('At least two message-passing layers are required')
        self.depth = layers
        self.layer1 = DirectedMeanLayer(input_dim, hidden)
        self.layer2 = DirectedMeanLayer(hidden, hidden)
        self.extra_layers = nn.ModuleList((DirectedMeanLayer(hidden, hidden) for _ in range(layers - 2)))
        self.dropout = nn.Dropout(dropout)
        self.head = nn.Sequential(nn.Linear(input_dim + hidden, hidden), nn.ReLU(), nn.Dropout(dropout), nn.Linear(hidden, 1))

    def forward(self, graph):
        h = self.dropout(self.layer1(graph.x, graph.incoming, graph.outgoing))
        h = self.dropout(self.layer2(h, graph.incoming, graph.outgoing))
        for layer in self.extra_layers:
            h = self.dropout(layer(h, graph.incoming, graph.outgoing))
        return self.head(torch.cat([graph.x, h], dim=1)).squeeze(-1)

def graph_batches(data, cut, end):
    raw = np.column_stack([data.node_raw, data.steps, np.log1p(data.indeg), np.log1p(data.outdeg)]).astype(np.float32)
    fit = (data.steps <= cut) & data.has_features
    scaler = StandardScaler().fit(raw[fit])
    x = scaler.transform(raw).astype(np.float32)
    x[~np.isfinite(x)] = 0
    x = np.column_stack([x, (~data.has_features).astype(np.float32)]).astype(np.float32)
    train = make_batch(data.steps <= cut, x, data.node_y, data.steps, data.ids, data.src, data.dst, DEVICE)
    valid = make_batch((data.steps > cut) & (data.steps <= end), x, data.node_y, data.steps, data.ids, data.src, data.dst, DEVICE)
    return (train, valid, scaler)

def fit_graph(data, cut, end, seed, profile, layers=2, epochs=None):
    seed_all(seed)
    train, valid, scaler = graph_batches(data, cut, end)
    model = DirectedGraphSAGE(train.x.shape[1], 128, 0.2, layers).to(DEVICE)
    opt = torch.optim.AdamW(model.parameters(), lr=0.001, weight_decay=0.0001)
    best = -np.inf
    stale = 0
    selected = 0
    curve = []
    for epoch in range(1, (epochs or profile['max_epochs']) + 1):
        model.train()
        opt.zero_grad(set_to_none=True)
        logits = model(train)
        y = train.y[train.labeled]
        weight = (y == 0).sum() / torch.clamp((y == 1).sum(), min=1)
        loss = F.binary_cross_entropy_with_logits(logits[train.labeled], y, pos_weight=weight)
        loss.backward()
        nn.utils.clip_grad_norm_(model.parameters(), 5)
        opt.step()
        row = dict(epoch=epoch, loss=float(loss.detach()))
        if epochs is None:
            model.eval()
            with torch.inference_mode():
                p = model(valid)[valid.labeled].sigmoid().cpu().numpy()
            row.update(metrics(valid.y[valid.labeled].cpu().numpy(), p))
            if row['auc'] > best:
                best = row['auc']
                selected = epoch
                stale = 0
            else:
                stale += 1
        else:
            selected = epoch
        curve.append(row)
        if epochs is None and stale >= profile['patience']:
            break
    model.eval()
    with torch.inference_mode():
        p = model(valid)[valid.labeled].sigmoid().cpu().numpy()
    weights = {k: v.cpu() for k, v in model.state_dict().items()}
    ids = valid.tx_ids[valid.labeled.cpu().numpy()]
    del model, opt, train, valid
    gc.collect()
    return (p, ids, selected, curve, weights, scaler)

def run_graph_trial(data, out, cut=28, horizon=7, seed=42, profile=None, layers=2, stage='graph_baselines'):
    profile = profile or profiles('quick')
    spec = dict(version=VERSION, data=data.identity, stage=stage, model='graphsage', variant=f'layers{layers}', cut=cut, horizon=horizon, seed=seed, profile=profile, config={'layers': layers}, inner_horizon=3, base='graph', kind='time', scaler='standard', engine_hash=globals().get('ENGINE_HASH', VERSION))
    job = Path(out) / stage / f'graphsage_{fingerprint(spec)[:16]}'
    job.mkdir(parents=True, exist_ok=True)
    if (job / 'complete.json').exists():
        return json.loads((job / 'complete.json').read_text())
    _, _, ep, selection, _, _ = fit_graph(data, cut - 3, cut, seed, profile, layers)
    p, ids, _, curve, weights, scaler = fit_graph(data, cut, cut + horizon, seed, profile, layers, ep)
    source = data.frame.set_index('txId').loc[ids].reset_index()
    pred = source[['txId', 'time_step', 'label']].copy()
    pred['prediction'] = p.astype(float)
    pred.to_csv(job / 'predictions.csv', index=False)
    torch.save(weights, job / 'model.pt')
    joblib.dump(scaler, job / 'preprocessor.joblib')
    save_json(job / 'selection.json', selection)
    save_json(job / 'refit_curve.json', curve)
    r = dict(spec=spec, metrics=metrics(pred.label, p), selected_epochs=ep, query_ids_hash=array_hash(ids), prediction_hash=file_hash(job / 'predictions.csv'), path=str(job), complete=True, outer_labels_used_for_epoch_selection=False)
    save_json(job / 'complete.json', r)
    return r

class GraphStore:
    """No labels are stored or consumed. Context may include unknown nodes."""

    def __init__(self, observed, edges):
        self.ids = observed.txId.to_numpy(np.int64)
        self.steps = observed.time_step.to_numpy(np.int64)
        if len(np.unique(self.ids)) != len(self.ids):
            raise ValueError('Duplicate observed node IDs')
        self.frame = observed.drop(columns=['label'], errors='ignore').copy()
        edges = edges[['txId1', 'txId2']].drop_duplicates()
        index = pd.Index(self.ids)
        src, dst = (index.get_indexer(edges.txId1), index.get_indexer(edges.txId2))
        total_in = self.frame.txId.map(edges.txId2.value_counts()).fillna(0).to_numpy()
        total_out = self.frame.txId.map(edges.txId1.value_counts()).fillna(0).to_numpy()
        known = (src >= 0) & (dst >= 0)
        src, dst = (src[known], dst[known])
        if not np.array_equal(self.steps[src], self.steps[dst]):
            raise ValueError('Cross-period observed edge: reject temporal partition leakage')
        n = len(self.ids)
        known_in, known_out = (np.bincount(dst, minlength=n), np.bincount(src, minlength=n))
        self.up = coo_matrix((1.0 / known_in[dst], (dst, src)), shape=(n, n)).tocsr()
        self.down = coo_matrix((1.0 / known_out[src], (src, dst)), shape=(n, n)).tocsr()
        self.coverage = np.column_stack([np.divide(known_in, total_in, out=np.zeros(n), where=total_in > 0), np.divide(known_out, total_out, out=np.zeros(n), where=total_out > 0)]).astype(np.float32)
        self.frame['in_degree'], self.frame['out_degree'] = (total_in, total_out)
        self.audit = dict(observed_nodes=n, known_edges=len(src), all_edges=len(edges), missing_source_edges=int((~known & (index.get_indexer(edges.txId2) >= 0)).sum()), missing_destination_edges=int((~known & (index.get_indexer(edges.txId1) >= 0)).sum()), labels_used=False, cross_period_edges=0, neighbor_sampling=False)

    def positions(self, ids):
        rows = pd.Index(self.ids).get_indexer(ids)
        if np.any(rows < 0):
            raise ValueError('Query node absent from supplied train feature rows')
        return rows

    def prepare_pool(self, names, bundle, device='cuda'):
        raw = self.frame[names].to_numpy(np.float32)
        keep = np.asarray(names) != 'feat_3'
        X = apply_num(raw[:, keep], bundle['numerical_preprocessor'])
        C = category_codes(self.frame.feat_3.to_numpy(np.float32), bundle['levels'])
        onehot = np.eye(bundle['cardinalities'][0], dtype=np.float32)[C]
        features = torch.from_numpy(np.column_stack([X, onehot])).to(device)
        return GraphPool(self, features, device)


In [ ]:
class GraphPool:

    def __init__(self, store, features, device):
        self.store, self.features, self.device = (store, features, device)
        self.coverage = torch.from_numpy(store.coverage).to(device)

    def batch(self, rows):
        up, down = (self.store.up[rows].tocoo(), self.store.down[rows].tocoo())
        unique = np.unique(np.concatenate([up.col, down.col]))
        neighbors = self.features[torch.as_tensor(unique, device=self.device)]
        matrices = []
        for part in [up, down]:
            indices = np.stack([part.row, np.searchsorted(unique, part.col)])
            matrices.append(torch.sparse_coo_tensor(torch.as_tensor(indices, dtype=torch.long, device=self.device), torch.as_tensor(part.data, dtype=torch.float32, device=self.device), (len(rows), len(unique)), device=self.device).coalesce())
        return (neighbors, matrices[0], matrices[1], self.coverage[torch.as_tensor(rows, device=self.device)])

class GatedGraphTabM(nn.Module):

    def __init__(self, tabm, numeric_dim, cardinality, config):
        super().__init__()
        self.tabm = tabm
        self.cardinality = cardinality
        width = config['graph_width']
        self.project = nn.Sequential(nn.Linear(numeric_dim + cardinality, width), nn.LayerNorm(width), nn.ReLU())
        self.context_residual = nn.Sequential(nn.Linear(width, width), nn.LayerNorm(width), nn.ReLU(), nn.Dropout(0.1))
        combined = width * 3 + 2
        self.correction = nn.Sequential(nn.Linear(combined, config['correction_hidden']), nn.ReLU(), nn.Dropout(0.1), nn.Linear(config['correction_hidden'], 1))
        self.gate = nn.Sequential(nn.Linear(combined, config['gate_hidden']), nn.ReLU(), nn.Linear(config['gate_hidden'], 1))
        nn.init.constant_(self.gate[-1].bias, config['gate_bias'])
        nn.init.zeros_(self.correction[-1].weight)
        nn.init.zeros_(self.correction[-1].bias)

    def encode(self, features):
        h = self.project(features)
        return h + self.context_residual(h)

    def forward(self, x, c, neighbors, upstream, downstream, coverage):
        base = self.tabm(x, c)
        node = self.encode(torch.cat([x, F.one_hot(c[:, 0], self.cardinality).to(x.dtype)], dim=1))
        neighbor_h = self.encode(neighbors)
        up, down = (torch.sparse.mm(upstream, neighbor_h), torch.sparse.mm(downstream, neighbor_h))
        combined = torch.cat([node, up, down, coverage], dim=1)
        gate = self.gate(combined).sigmoid() * (coverage.sum(1, keepdim=True) > 0)
        correction = gate * self.correction(combined)
        delta = torch.cat([torch.zeros_like(correction), correction], dim=1)[:, None, :]
        return (base + delta, base, gate[:, 0])

def fit_hybrid(data, fit, query, seed, profile, epochs=None, val_y=None):
    seed_all(seed)
    X, V, C, D, bundle = make_inputs(data, fit, query, 'cat3')
    store = GraphStore(data.train, data.edges)
    raw = store.frame[FEATURES + ['time_step', 'in_degree', 'out_degree']].to_numpy(np.float32)
    numeric = apply_num(np.delete(raw, 2, 1), bundle['numerical_preprocessor'])
    code = category_codes(store.frame.feat_3.to_numpy(np.float32), bundle['levels'])
    cardinality = bundle['cardinalities'][0]
    pool = GraphPool(store, torch.as_tensor(np.column_stack([numeric, np.eye(cardinality)[code]]), dtype=torch.float32, device=DEVICE), DEVICE)
    model = GatedGraphTabM(build_net('tabm', X, [cardinality], TABM), X.shape[1], cardinality, dict(graph_width=64, correction_hidden=128, gate_hidden=32, gate_bias=-2)).to(DEVICE)
    opt = torch.optim.AdamW(model.parameters(), lr=0.002, weight_decay=0.0003)
    tx = torch.as_tensor(X, device=DEVICE)
    tc = torch.as_tensor(C, device=DEVICE)
    ty = torch.as_tensor(data.y[fit], device=DEVICE)
    fitrows = store.positions(data.frame.txId.iloc[fit])
    queryrows = store.positions(data.frame.txId.iloc[query])

    def infer():
        model.eval()
        parts = []
        baseparts = []
        gates = []
        with torch.inference_mode():
            for start in range(0, len(V), profile['batch_size']):
                stop = start + profile['batch_size']
                context = pool.batch(queryrows[start:stop])
                logits, base, g = model(torch.as_tensor(V[start:stop], device=DEVICE), torch.as_tensor(D[start:stop], device=DEVICE), *context)
                parts.append(logits.softmax(-1)[..., 1].mean(1).cpu().numpy())
                baseparts.append(base.softmax(-1)[..., 1].mean(1).cpu().numpy())
                gates.append(g.cpu().numpy())
        return (np.concatenate(parts), np.concatenate(baseparts), np.concatenate(gates))
    best = -np.inf
    stale = 0
    selected = 0
    curve = []
    for epoch in range(1, (epochs or min(profile['max_epochs'], 60)) + 1):
        model.train()
        total = 0.0
        for indices in torch.randperm(len(X)).split(profile['batch_size']):
            ix = indices.numpy()
            opt.zero_grad(set_to_none=True)
            logits, _, _ = model(tx[indices.to(DEVICE)], tc[indices.to(DEVICE)], *pool.batch(fitrows[ix]))
            loss = head_loss(logits, ty[indices.to(DEVICE)])
            loss.backward()
            nn.utils.clip_grad_norm_(model.parameters(), 1)
            opt.step()
            total += float(loss.detach()) * len(ix)
        row = dict(epoch=epoch, loss=total / len(X))
        if epochs is None:
            p, _, _ = infer()
            row.update(metrics(val_y, p))
            if row['auc'] > best:
                best = row['auc']
                selected = epoch
                stale = 0
            else:
                stale += 1
        else:
            selected = epoch
        curve.append(row)
        if epochs is None and stale >= profile['patience']:
            break
    p, base, gates = infer()
    weights = {k: v.cpu() for k, v in model.state_dict().items()}
    del model, opt, pool
    gc.collect()
    return (p, selected, curve, weights, bundle, base, gates)

def run_hybrid_trial(data, out, cut=21, horizon=14, seed=42, profile=None, stage='appendix_neural'):
    profile = profile or profiles('quick')
    spec = dict(version=VERSION, data=data.identity, stage=stage, model='graph_tabm', variant='cat3', cut=cut, horizon=horizon, seed=seed, profile=profile, config=TABM, inner_horizon=3, base='tabm', kind='time', scaler='quantile', engine_hash=globals().get('ENGINE_HASH', VERSION))
    job = Path(out) / stage / f'graph_tabm_{fingerprint(spec)[:16]}'
    job.mkdir(parents=True, exist_ok=True)
    if (job / 'complete.json').exists():
        return json.loads((job / 'complete.json').read_text())
    a, b, ia, ib = split_indices(data, cut, horizon)
    _, ep, selection, _, _, _, _ = fit_hybrid(data, ia, ib, seed, profile, val_y=data.y[ib])
    p, _, curve, weights, bundle, base, gates = fit_hybrid(data, a, b, seed, profile, epochs=ep)
    pred = data.frame.iloc[b][['txId', 'time_step', 'label']].copy()
    pred['prediction'] = p.astype(float)
    pred.to_csv(job / 'predictions.csv', index=False)
    np.savez_compressed(job / 'correction_ablation.npz', enabled=p, disabled=base, gate=gates)
    torch.save(weights, job / 'model.pt')
    joblib.dump(bundle, job / 'preprocessing.joblib')
    save_json(job / 'selection.json', selection)
    save_json(job / 'refit_curve.json', curve)
    r = dict(spec=spec, metrics=metrics(data.y[b], p), disabled_metrics=metrics(data.y[b], base), selected_epochs=ep, query_ids_hash=array_hash(pred.txId.to_numpy()), prediction_hash=file_hash(job / 'predictions.csv'), path=str(job), complete=True)
    save_json(job / 'complete.json', r)
    return r

def run_realmlp_trial(data, out, cut=21, horizon=14, seed=42, profile=None, stage='appendix_neural'):
    """Original official RealMLP-TD API, chronological inner fit then explicit no-validation refit."""
    from pytabkit import RealMLP_TD_Classifier
    profile = profile or profiles('quick')
    a, b, ia, ib = split_indices(data, cut, horizon)

    def arrays(fit, query):
        names = FEATURES + ['time_step', 'in_degree', 'out_degree']
        numeric = [n for n in names if n != 'feat_3']
        frame = data.frame.copy()
        frame['in_degree'] = data.indeg[data.pos]
        frame['out_degree'] = data.outdeg[data.pos]
        levels = np.unique(frame.feat_3.iloc[fit])
        med = frame[numeric].iloc[fit].median()
        A = frame[numeric].iloc[fit].fillna(med).copy()
        B = frame[numeric].iloc[query].fillna(med).copy()
        A['feat_3'] = category_codes(frame.feat_3.iloc[fit].to_numpy(), levels)
        B['feat_3'] = category_codes(frame.feat_3.iloc[query].to_numpy(), levels)
        return (A, B, dict(levels=levels, medians=med.to_dict()))
    config = dict(n_epochs=2 if profile['mode'] == 'quick' else 256, batch_size=256, predict_batch_size=256, n_cv=1, n_refit=0, use_ls=False, val_metric_name='1-auc_ovr', use_early_stopping=True, early_stopping_additive_patience=10, early_stopping_multiplicative_patience=1.0, device='cuda:0' if DEVICE.type == 'cuda' else 'cpu', random_state=seed, n_threads=2, verbosity=0)
    spec = dict(version=VERSION, data=data.identity, stage=stage, model='realmlp', variant='cat3', cut=cut, horizon=horizon, seed=seed, profile=profile, config=config, inner_horizon=3, base='tabm', kind='time', scaler='RealMLP_TD', engine_hash=globals().get('ENGINE_HASH', VERSION))
    job = Path(out) / stage / f'realmlp_{fingerprint(spec)[:16]}'
    job.mkdir(parents=True, exist_ok=True)
    if (job / 'complete.json').exists():
        return json.loads((job / 'complete.json').read_text())
    A, B, ip = arrays(ia, ib)
    inner = RealMLP_TD_Classifier(**config)
    inner.fit(A, data.y[ia], B, data.y[ib], cat_col_names=['feat_3'])
    ep = int(inner.fit_params_['stop_epoch']['1-auc_ovr'])
    del inner
    X, V, op = arrays(a, b)
    model = RealMLP_TD_Classifier(**dict(config, stop_epoch=ep, use_early_stopping=False))
    model.fit(X, data.y[a], val_idxs=np.empty(0, np.int64), cat_col_names=['feat_3'])
    p = model.predict_proba(V)[:, 1]
    import cloudpickle
    model.to('cpu')
    (job / 'model.pkl').write_bytes(cloudpickle.dumps(dict(model=model, outer=op, inner=ip)))
    pred = data.frame.iloc[b][['txId', 'time_step', 'label']].copy()
    pred['prediction'] = p.astype(float)
    pred.to_csv(job / 'predictions.csv', index=False)
    r = dict(spec=spec, metrics=metrics(data.y[b], p), selected_epochs=ep, query_ids_hash=array_hash(pred.txId.to_numpy()), prediction_hash=file_hash(job / 'predictions.csv'), path=str(job), complete=True)
    save_json(job / 'complete.json', r)
    return r


In [ ]:
P = profiles(PROFILE)


## 3. Load and check the data


In [ ]:
# Labelled rows are the only rows used by supervised models.
data = Dataset(DATA_DIR, expected_competition=EXPECTED_COMPETITION)
print('Labelled training rows:', len(data.frame), '| Unknown labels:', int(data.train.label.isna().sum()), '| Test rows:', len(data.test))


## 4. Linear, tree and neural baselines
Linear and tree models use 165 features plus time_step. TabM and TabR also use in/out degree. These are complete-pipeline comparisons.


In [ ]:
BASELINE_MODELS = globals().get('BASELINE_MODELS', ['logistic','lightgbm','tabm'] if PROFILE == 'quick' else
    ['logistic','linear','ridge','ridge_classifier','random_forest','extra_trees','hist_gradient_boosting',
     'xgboost','lightgbm','catboost','mlp','resnet','ft_transformer','lstm','tabm','tabr'])
for model in BASELINE_MODELS:
    base = 'tabm' if model in ['tabm','tabr'] else 'table'
    for seed in P['seeds']:
        run_trial(data, OUT, model=model, cut=28, horizon=7, seed=seed, profile=P,
                  base=base, stage='initial_baselines')
baseline = aggregate_runs(prediction_records(OUT, 'initial_baselines'))
display(baseline.sort_values('auc', ascending=False))
baseline.to_csv(OUT / 'initial_baselines.csv', index=False)


## 5. Directional GraphSAGE
Optionally compare two and six message-passing layers. Loss is computed only for labelled nodes.


In [ ]:
RUN_GNN = globals().get('RUN_GNN', False)
if RUN_GNN:
    for layers in [2, 6]:
        for seed in P['seeds']:
            run_graph_trial(data, OUT, cut=28, horizon=7, seed=seed, profile=P, layers=layers)
    display(aggregate_runs(prediction_records(OUT, 'graph_baselines')))
else:
    print('GraphSAGE skipped. Set RUN_GNN=True to run it.')


## 6. Additional neural candidates
Optionally evaluate the categorical FT Transformer, gated Graph-TabM and official RealMLP implementation from Appendix E. RealMLP requires its own optional installation.


In [ ]:
RUN_GRAPH_HYBRID = globals().get('RUN_GRAPH_HYBRID', False)
RUN_REALMLP = globals().get('RUN_REALMLP', False)
if RUN_REALMLP:
    subprocess.run([sys.executable,'-m','pip','install','-q','pytabkit==1.7.3','cloudpickle'], check=True)
if RUN_GRAPH_HYBRID or RUN_REALMLP:
    for cutoff in ([21] if PROFILE == 'quick' else [14,21]):
        for seed in P['seeds']:
            for model in ['tabm','ft_transformer']:
                run_trial(data, OUT, model=model, variant='cat3', cut=cutoff, seed=seed,
                    profile=P, stage='appendix_neural')
            if RUN_GRAPH_HYBRID:
                run_hybrid_trial(data, OUT, cut=cutoff, seed=seed, profile=P)
            if RUN_REALMLP:
                run_realmlp_trial(data, OUT, cut=cutoff, seed=seed, profile=P)
    display(aggregate_runs(prediction_records(OUT, 'appendix_neural')))
else:
    print('Optional additional neural candidates skipped.')


## 7. Three-boosting-model ridge baseline
This optional baseline uses XGBoost, LightGBM and CatBoost. Fit a ridge meta-model on unique OOF predictions from periods 8-28 and score periods 29-35. It is separate from the final TabM/tree logistic stacker in notebook 07.


In [ ]:
RUN_RIDGE_STACK = globals().get('RUN_RIDGE_STACK', False)
if RUN_RIDGE_STACK:
    matrices = {}
    for cutoff in [7,14,21,28]:
        columns, reference = [], None
        for model in ['xgboost','lightgbm','catboost']:
            parts = []
            for seed in P['seeds']:
                record = run_trial(data, OUT, model=model, cut=cutoff, horizon=7, seed=seed,
                    profile=P, base='table', stage='ridge_baseline_oof')
                frame = pd.read_csv(Path(record['path']) / 'predictions.csv', float_precision='round_trip')
                if reference is None:
                    reference = frame[['txId','time_step','label']].copy()
                else:
                    np.testing.assert_array_equal(reference, frame[['txId','time_step','label']])
                parts.append(frame.prediction.to_numpy())
            columns.append(np.mean(parts, axis=0))
        matrices[cutoff] = reference, np.column_stack(columns)
    meta_rows = pd.concat([matrices[k][0] for k in [7,14,21]], ignore_index=True)
    assert meta_rows.txId.is_unique
    M = np.vstack([matrices[k][1] for k in [7,14,21]])
    target, Q = matrices[28]
    assert not set(meta_rows.txId) & set(target.txId)
    scaler = StandardScaler().fit(M)
    stacker = Ridge(alpha=1).fit(scaler.transform(M), meta_rows.label)
    result = metrics(target.label, stacker.predict(scaler.transform(Q)))
    save_json(OUT / 'ridge_baseline_metrics.json', result)
    print(result)
else:
    print('Optional three-boosting ridge baseline skipped.')
